# Install

In [ ]:
if False:
    !sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
    !sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet gspread google-api-python-client google-auth-httplib2 google-auth-oauthlib)"
    !sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet llm)"

# Import

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [12]:
import logging
import os

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint

print(henv.get_system_signature()[0])

hnotebook.config_notebook()

# hdbg.init_logger(verbosity=logging.DEBUG)
hdbg.init_logger(verbosity=logging.INFO)
# hdbg.test_logger()
_LOG = logging.getLogger(__name__)

# System signature
  # Container version
    container_version='2.0.0'
    changelog_version='2.2.0'
  # Git info
    branch_name='CsfyTask7819_Add_blogs'
    hash='71a96d051'
    # Last commits:
      * 71a96d051 GP Saggese Update                                                            (25 minutes ago) Tue Dec 16 20:32:37 2025  (HEAD -> CsfyTask7819_Add_blogs, origin/CsfyTask7819_Add_blogs)
      * 3b17e2a54 GP Saggese Update                                                            (   5 hours ago) Tue Dec 16 16:08:28 2025           
      *   459be8bf8 GP Saggese Update                                                            (   6 hours ago) Tue Dec 16 15:25:33 2025           
      |\  
  # Platform info
    system=Linux
    node name=581253e6cff0
    release=6.10.14-linuxkit
    version=#1 SMP Tue Apr 15 16:00:54 UTC 2025
    machine=aarch64
    processor=aarch64
  # psutils info
    cpu count=8
    cpu freq=None
    memory=svmem(total=16749285376, available=13721837568, pe

In [13]:
import importlib

import ck_marketing.workflows as ckmktwf

importlib.reload(ckmktwf)

import ck_marketing.plugins as ckmktpi

importlib.reload(ckmktpi)

import helpers.hgoogle_drive_api as hgodrapi

# Get credentials first
credentials = hgodrapi.get_credentials(
    service_key_path="/home/.config/gspread_pandas/google_secret.json"
)

# Load data

In [15]:
# df = pb.load_49_companies_data()
# df = pb.load_200_companies_data()

# Pitchbook results for 200AICompanies.MA
url = "https://docs.google.com/spreadsheets/d/1aKzWUw9mwP-2_vzz27ggeLe1sgF9OrWRWmGMo9Dk9bU/edit?gid=0#gid=0"
tab_name = "200AICompanies.MA"
df = ckmktpi.get_pitchbook_data_from_gsheet(url, tab_name)

hpandas.head(df)

columns= ['idx', 'People', 'LinkedInURL', 'LastName', 'FirstName', 'PrimaryCompany', 'PrimaryPosition', 'Biography', 'BoardSeats', 'Roles', 'DealRoles', 'Location', 'AddressLine1', 'AddressLine2', 'City', 'State/Province', 'PostCode', 'Country/Territory/Region', 'Phone', 'Fax', 'Email']
shape= (718, 21)


,idx,People,LinkedInURL,LastName,FirstName,PrimaryCompany,PrimaryPosition,Biography,BoardSeats,Roles,DealRoles,Location,AddressLine1,AddressLine2,City,State/Province,PostCode,Country/Territory/Region,Phone,Fax,Email
0,0,Oliver Steinig,http://linkedin.com/in/oliver-steinig-68603aa/,Steinig,Oliver,Bosch (Automotive),Vice President of Business Development and Cor...,Ms. Oliver Steinig serves as Vice President of...,,1,,"Gerlingen, Germany",Robert-Bosch-Platz 1,Schillerh��he,Gerlingen,,70839,Germany,#ERROR!,#ERROR!,oliver.steinig@bosch.com
1,1,Harrick Vin Ph.D,http://linkedin.com/in/harrickvin,Vin,Harrick,Tata Consultancy Services (Mumbai),Chief Technology Officer,Dr. Harrick Vin serves as Chief Technology Off...,,1,,"Mumbai, India",9th Floor Nirmal Building,Nariman Point,Mumbai,Maharashtra,400021,India,#ERROR!,#ERROR!,harrick.vin@tcs.com


# Clean up

## Remove invalid emails

In [16]:
df = ckmktpi.remove_invalid_emails(df, verbose=False)
hpandas.head(df)

INFO  Removed emails: 47 / 718 = 6.55%
columns= ['idx', 'People', 'LinkedInURL', 'LastName', 'FirstName', 'PrimaryCompany', 'PrimaryPosition', 'Biography', 'BoardSeats', 'Roles', 'DealRoles', 'Location', 'AddressLine1', 'AddressLine2', 'City', 'State/Province', 'PostCode', 'Country/Territory/Region', 'Phone', 'Fax', 'Email']
shape= (671, 21)


,idx,People,LinkedInURL,LastName,FirstName,PrimaryCompany,PrimaryPosition,Biography,BoardSeats,Roles,DealRoles,Location,AddressLine1,AddressLine2,City,State/Province,PostCode,Country/Territory/Region,Phone,Fax,Email
0,0,Oliver Steinig,http://linkedin.com/in/oliver-steinig-68603aa/,Steinig,Oliver,Bosch (Automotive),Vice President of Business Development and Cor...,Ms. Oliver Steinig serves as Vice President of...,,1,,"Gerlingen, Germany",Robert-Bosch-Platz 1,Schillerh��he,Gerlingen,,70839,Germany,#ERROR!,#ERROR!,oliver.steinig@bosch.com
1,1,Harrick Vin Ph.D,http://linkedin.com/in/harrickvin,Vin,Harrick,Tata Consultancy Services (Mumbai),Chief Technology Officer,Dr. Harrick Vin serves as Chief Technology Off...,,1,,"Mumbai, India",9th Floor Nirmal Building,Nariman Point,Mumbai,Maharashtra,400021,India,#ERROR!,#ERROR!,harrick.vin@tcs.com


In [17]:
df.iloc[0]

idx                                                                         0
People                                                         Oliver Steinig
LinkedInURL                    http://linkedin.com/in/oliver-steinig-68603aa/
LastName                                                              Steinig
FirstName                                                              Oliver
PrimaryCompany                                             Bosch (Automotive)
PrimaryPosition             Vice President of Business Development and Cor...
Biography                   Ms. Oliver Steinig serves as Vice President of...
BoardSeats                                                                   
Roles                                                                       1
DealRoles                                                                    
Location                                                   Gerlingen, Germany
AddressLine1                                             Robert-

## Validate emails 

In [18]:
os.environ["HUNTER_API_KEY"] = "c5e3c68e63e52a0e2ca088e8c73a95220cd5ba48"

In [19]:
ckmktpi.get_hunterio_account_info()

{'credits': {'used': 283.0, 'available': 2000.0},
 'searches': {'used': 283, 'available': 2000},
 'verifications': {'used': 566, 'available': 4000},
 'reset_date': '2026-01-13'}

In [20]:
# first_name = "Boone"
# first_name = "Keith"
# company_name = "Accenture"
# # email = "toby.cappello@ibm.com"
# # email = "csolder@cisco.com"
# # email = "kb@accenture.com"
# # email = "alois.reitbauer@dynatrace.com"
# email = "vaibhav.narayanam@servicenow.com"

# ckmktpi.verify_email(email)

In [21]:
# url = "https://docs.google.com/spreadsheets/d/1HyglraD02TJwp16wkU_yZZ6jJW51LnBdlaWsVSTNLws"
# tab_name = "Sheet4"
# df_tmp = hgodrapi.from_gsheet(credentials, url, tab_name=tab_name)
# hpandas.head(df_tmp)

In [22]:
df2 = ckmktpi.hunterio_verify_emails_from_df(df, email_col="Email")
hpandas.head(df2)

  0%|          | 0/671 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
df2["hunterio.email_verification"].unique()

In [ ]:
# Print stats.
# stats_dict = ckmktpi.get_email_stats(df2, "hunterio.email_verification", is_email_verification=True)
# pprint.pprint(stats_dict)

In [ ]:
df3 = ckmktpi.remove_invalid_emails_from_hunterio_df(df2)

hpandas.head(df3)

## Process

In [ ]:
df4 = ckmktpi.process_pitchbook_dataframe(df3, add_role_columns=False)
hpandas.head(df4)

In [ ]:
# df5 = ckmktpi.pick_people(df4)
df5 = df4

In [ ]:
# df2["Biography"].iloc[0]

In [ ]:
# url = "https://docs.google.com/spreadsheets/d/1LYC46zrYJMpljaofxrBdrEDidwNSV6DwtljEzOxN8eY/edit?gid=0#gid=0"

# tmp
url = "https://docs.google.com/spreadsheets/d/1zB3p-bgRm_WDQbdMK-nfMeDDcrFtQI7YME4C5TsTBYU"
hgodrapi.to_gsheet(
    df5, url, tab_name="before", freeze_rows=True, credentials=credentials
)

In [ ]:
# OpenAI client logging
logging.getLogger("openai").setLevel(logging.WARNING)

# Common HTTP logging sources
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [ ]:
## Create jobs

In [ ]:
# df6 = ckmktpi.apply_func_to_df(df5.iloc[0:100], ckmktpi.get_job_word, "job_word")
df6 = ckmktpi.apply_func_to_df(df5.iloc[100:], ckmktpi.get_job_word, "job_word")

hpandas.head(df6)

In [ ]:
print(ckmktpi.get_LIn_connect_message(df6.iloc[0]))

In [ ]:
df7 = ckmktpi.apply_func_to_df(df6, ckmktpi.get_LIn_connect_message, "LIn_msg")

hpandas.head(df7)

In [ ]:
# url = "https://docs.google.com/spreadsheets/d/1LYC46zrYJMpljaofxrBdrEDidwNSV6DwtljEzOxN8eY/edit?gid=0#gid=0"

# tmp
url = "https://docs.google.com/spreadsheets/d/1zB3p-bgRm_WDQbdMK-nfMeDDcrFtQI7YME4C5TsTBYU"
hgodrapi.to_gsheet(
    df7, url, tab_name="before", freeze_rows=True, credentials=credentials
)

## Compute stats

In [ ]:
df2.head()

In [ ]:
pb.compute_pitchbook_stats(df2, top_n=10)

In [ ]:
df3 = df2.sample(frac=1, random_state=42)

In [ ]:
df3.head()

In [ ]:
assert 0

# Compute emails

In [ ]:
idx = 0
first_name = df3.iloc[idx]["FirstName"]
# company_website = df3.iloc[idx]["PrimaryCompany Website"]
company_website = ""
company_name = df3.iloc[idx]["PrimaryCompany"]
print(first_name, company_website, company_name)

response = pb.get_email2(first_name, company_name)
print(response)
# response = pb.get_email2(company_website, company_name)
# print(response)

# response = pb.get_business_descr(company_website, company_name)
# print(response)

In [ ]:
from tqdm.auto import tqdm  # or: from tqdm import tqdm

emails = []

df4 = df3.iloc[150:200]
# df4 = df3.iloc[101:]

_LOG.setLevel(0)
for idx, row in tqdm(df4.iterrows(), total=len(df4)):
    first_name = row["FirstName"]
    # company_website = row["PrimaryCompany Website"]
    company_name = row["PrimaryCompany"]

    # print(company_website, company_name)  # optional
    # response = pb.get_email(first_name, company_website, company_name)
    # response = pb.get_email3(first_name, company_website, company_name)
    response = pb.get_email2(first_name, company_name)
    # print(response)  # optional

    emails.append(response)
_LOG.setLevel(logging.INFO)

df4["EmailText"] = emails

In [ ]:
df4

In [ ]:
print(df4.iloc[2]["EmailText"])

In [ ]:
if False:
    # df5 = df4.copy()
    df5 = df4

    df5["EmailText"] = df5["EmailText"].apply(
        lambda x: "\n".join(x.splitlines()[:-3])
    )

In [ ]:
print(df5.iloc[2]["EmailText"])

In [ ]:
url = "https://docs.google.com/spreadsheets/d/1zB3p-bgRm_WDQbdMK-nfMeDDcrFtQI7YME4C5TsTBYU/edit?gid=604528158#gid=604528158"
hgodrapi.to_gsheet(df4, url, tab_name="final", credentials=credentials)

# LinkedIn

In [ ]:
assert 0

In [ ]:
linkedin_msg = """Hi {FirstName},

This is GP, and I teach AI at the Univ of Maryland.

I work on Causal AI, helping LLMs understand time and probability. We’ve reached $1M ARR in 12 months. 

Given your work at {PrimaryCompany}, I thought this might be relevant. Can I tell you more about it?

Thanks,
GP"""

# Assuming your dataframe is named df
df_all = df2.copy()
df2["LinkedInMessage"] = df2.apply(
    lambda row: linkedin_msg.format(
        FirstName=row["FirstName"], PrimaryCompany=row["PrimaryCompany"]
    ),
    axis=1,
)
print(df2.shape)

In [ ]:
txt = df2.iloc[0]["LinkedInMessage"]
print(txt)
print(len(txt))

In [ ]:
df_all.head(2)

In [ ]:
df_all = df_all[
    "People	LinkedInURL	PrimaryCompany	FirstName	LinkedInMessage".split()
]

In [ ]:
url = "https://docs.google.com/spreadsheets/d/1zB3p-bgRm_WDQbdMK-nfMeDDcrFtQI7YME4C5TsTBYU/edit?gid=604528158#gid=604528158"
hgodrapi.to_gsheet(df_all, url, tab_name="final", credentials=credentials)